# Frog OpenBCI - Pressure with Random Forest Regressor

# Get data
## Train data

In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

USE_BANDPASS = 1
window_size_seconds = 1.5 # 1.75
USE_NOTCH = 1
USE_ENVELOPE = 0
USE_KALMAN = 0
USE_HILBERT = 1
USE_ZSCORE = 0 
USE_SCALING = 1 
USE_TKEO = 0
#USE_NORMALIZATION = 0

sampling_rate = 250  # Hz

# USE_ALIGNZERO_GLOBAL = Substract the mean to each channel
# USE_ALIGNZERO_LOCAL 
# USE_OFFSET = Generate more windows with offset
# offset 
# USE_BANDPASS = Filter with bandpass (window by window)
# USE_ZSCORE = Filter with zscore (window by window)
# USE_SCALING = Use feature scaling

print("Settings updated")

samples_per_window = int(window_size_seconds * sampling_rate)
all_windowed_data = defaultdict(list)

users = range(1, 25)

for user in users:
    print(f"\nProcessing user {user}")
    
    emg_dir = fr"C:\Quick_Disk\OpenBCI_user_data\emg_data_OpenBCI\Participant_{user}\Pressure_data_user_{user}.csv"
    timestamp_dir = fr"C:\Quick_Disk\OpenBCI_user_data\Timestamps\Timestamps_pressure_user_{user}.csv"
    
    emg_df = pd.read_csv(emg_dir, sep='\t')  
    timestamp_df = pd.read_csv(timestamp_dir, sep=',')  
    
    emg_timestamps = pd.to_numeric(emg_df.iloc[:, 22], errors='coerce').values
    labels = timestamp_df.iloc[:, 0].values
    label_timestamps = pd.to_numeric(timestamp_df.iloc[:, 1], errors='coerce').values
    
    # Window data for this user
    windowed_data = defaultdict(list)
    
    for label, ts in zip(labels, label_timestamps):
        idx = np.argmin(np.abs(emg_timestamps - ts)) # index in the emg data
        
        if idx + samples_per_window <= len(emg_df): # Check if there is enough information for this window
            window_df = emg_df.iloc[idx:idx + samples_per_window, 1:4]
            window_array = window_df.to_numpy()
            windowed_data[label].append(window_array)
    
    # Merge into global dataset
    for label, windows in windowed_data.items():
        all_windowed_data[label].extend(windows)

print("\nAcquired data for all users")
for label, windows in all_windowed_data.items():
    print(f"  Class '{label}': {len(windows)} windows")

Settings updated

Processing user 1

Processing user 2

Processing user 3

Processing user 4

Processing user 5

Processing user 6

Processing user 7

Processing user 8

Processing user 9

Processing user 10

Processing user 11

Processing user 12

Processing user 13

Processing user 14

Processing user 15

Processing user 16

Processing user 17

Processing user 18

Processing user 19

Processing user 20

Processing user 21

Processing user 22

Processing user 23

Processing user 24

Acquired data for all users
  Class 'l0': 240 windows
  Class 'l25': 240 windows
  Class 'l50': 241 windows
  Class 'l100': 241 windows
  Class 'f0': 240 windows
  Class 'f25': 240 windows
  Class 'f50': 241 windows
  Class 'f100': 241 windows
  Class 'r0': 240 windows
  Class 'r25': 241 windows
  Class 'r50': 245 windows
  Class 'r100': 248 windows


# Process windows


## Filter windows

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch, hilbert, find_peaks
from pykalman import KalmanFilter

def bandpass_filter(data, lowcut=5.0, highcut=120.0, fs=sampling_rate, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def high_pass_filter(signal, cutoff=0.1, fs=sampling_rate, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def notch_filter(signal, freq=50.0, fs=sampling_rate, quality=30):
    nyquist = 0.5 * fs
    norm_freq = freq / nyquist
    b, a = iirnotch(norm_freq, quality)
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def compute_envelope(signal, fs=250, cutoff=5.0):
    rectified = np.abs(signal)
    b, a = butter(4, cutoff / (0.5 * fs), btype='low')
    envelope = filtfilt(b, a, rectified)
    return envelope

def compute_envelope_peaks(signal, min_peak_distance=10):
    # Find peaks
    peaks, _ = find_peaks(signal, distance=min_peak_distance)

    if len(peaks) < 2:
        # If not enough peaks, fallback to rectified signal or zeros
        print("Not enough peaks to compute envelope. Returning zeros.")
        return np.zeros_like(signal)

    # Extract peak values
    peak_values = signal[peaks]

    # Interpolate envelope
    envelope = np.interp(np.arange(len(signal)), peaks, peak_values)

    return envelope

def tkeo(signal):
    # Teager-Kaiser Energy Operator 
    output = np.zeros_like(signal)
    for i in range(1, len(signal) - 1):
        output[i] = signal[i]**2 - signal[i - 1] * signal[i + 1]  
    return output 

def kalman(signal):
    signal = signal.reshape(-1, 1)  # reshape to (n_samples, n_dim)
    kf = KalmanFilter(transition_matrices=[1],
                  observation_matrices=[1],
                  initial_state_mean=0,
                  observation_covariance=0.01,
                  transition_covariance=1e-5)

    state_means, _ = kf.filter(signal.reshape(-1, 1))

    return state_means.flatten()

def hilbert_envelope(signal):
    analytic = hilbert(signal)
    envelope = np.abs(analytic)
    return envelope

def normalization_max_val(signal, max_val):
    return signal / max_val if max_val != 0 else signal

def maximum_absolute_value(windowed_data):
    max_vals = np.zeros(3)
    for class_label, windows in windowed_data.items():
        for window in windows:
            abs_window = np.abs(window)
            max_vals = np.maximum(max_vals, abs_window.max(axis=0))
    return max_vals.tolist()

def filter_dataset(dataset):
    filtered_dataset = {}

    for class_label, windows in dataset.items():
        if not windows:
            print(f"No windows for class '{class_label}', skipping filter.")
            continue

        fwindows = []
        for window in windows:
            one_window = window.copy()
            for i in range(window.shape[1]):
                if i in [0, 1, 2]:  
                    if USE_NOTCH == 1:
                        one_window[:, i] = notch_filter(one_window[:, i])

                    if USE_BANDPASS == 1:
                        one_window[:, i] = bandpass_filter(one_window[:, i])

                    if USE_HILBERT == 1:
                        one_window[:, i] = hilbert_envelope(one_window[:, i])

                    if USE_KALMAN == 1:
                        one_window[:, i] = kalman(one_window[:, i])
                        print("Applied Kalman filter")

                    if USE_TKEO == 1:
                        one_window[:, i] = tkeo(one_window[:, i])      

                    if USE_ENVELOPE == 1:
                        one_window[:, i] = compute_envelope(one_window[:, i])

                    if USE_ZSCORE == 1:
                        mean = one_window[:, i].mean()
                        std = one_window[:, i].std()
                        if std == 0:
                            std = 1
                        one_window[:, i] = (one_window[:, i] - mean) / std

            fwindows.append(one_window)

        filtered_dataset[class_label] = fwindows

    return filtered_dataset


def normalize_filtered_data(filtered_data):
    max_vals = maximum_absolute_value(filtered_data)
    print(f"Global maximum absolute values: {max_vals}")

    for class_label, windows in filtered_data.items():
        for window in windows:
            for i in range(window.shape[1]):
                if i in [0, 1, 2]:
                    window[:, i] = normalization_max_val(window[:, i], max_vals[i])

    return filtered_data


filtered = []
filtered = filter_dataset(all_windowed_data)
print("Completed")

#if USE_NORMALIZATION == 1:
#    print("\nNormalized")
#    filtered = normalize_filtered_data(filtered)



Completed


## Features

In [16]:
import numpy as np
import pandas as pd
import pywt

# -------------- Feature functions 
def rms(signal):
    return np.sqrt(np.mean(signal**2))

def zero_crossings(signal):
    signs = np.signbit(signal)
    return np.sum(signs[1:] != signs[:-1])

def waveform_length(signal):
    return np.sum(np.abs(np.diff(signal)))

def mav(signal):
    return np.mean(np.abs(signal))

def iav(signal):
    return np.sum(np.abs(signal))

def rms_signed_difference(signal):
    mean_val = np.mean(signal)
    diff = signal - mean_val
    return np.sqrt(np.mean(diff**2))

def mean_frequency(signal, fs=250):
    # Compute FFT
    freqs = np.fft.rfftfreq(len(signal), d=1/fs)
    fft_vals = np.abs(np.fft.rfft(signal))
    power = fft_vals ** 2
    if np.sum(power) == 0:
        return 0
    mf = np.sum(freqs * power) / np.sum(power)
    return mf

def wavelet_features(signal, wavelet='db4', level=3):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    features = []
    for c in coeffs:
        energy = np.sum(np.square(c))
        std = np.std(c)
        features.extend([energy, std])
    return features

# -------------------

def parse_pressure_label(label):
    """
    Convert a label like 'l50' into a [left, right, front] pressure vector.
    
    Args:
        label (str): One of 'lXX', 'rXX', or 'fXX', where XX is the pressure (0–100)
    
    Returns:
        List[int]: [left_pressure, right_pressure, front_pressure]
    """
    pressures = {'l': 0, 'f': 0, 'r': 0}
    side = label[0]       # 'l', 'r', or 'f'
    value = int(label[1:]) 
    pressures[side] = value
    return [pressures['l'], pressures['f'], pressures['r']]


channels = ['ch_1', 'ch_2', 'ch_3']
feature_names = ['RMS', 'RMS_SD', 'ZC', 'WL', 'MAV', 'STD', 'VAR', 'IAV', 'MF']
#wavelet_feature_names = [f'W_E_L{i}' for i in range(4)] + [f'W_STD_L{i}' for i in range(4)]
#feature_names = feature_names + wavelet_feature_names

def extract_features(filtered_data):
    features = []
    targets = []

    for label, windows in filtered_data.items():
        for window in windows:
            feats = []
            for ch_idx in range(len(channels)):
                ch_signal = window[:, ch_idx]

                feats.append(rms(ch_signal))
                feats.append(rms_signed_difference(ch_signal))
                feats.append(zero_crossings(ch_signal))
                feats.append(waveform_length(ch_signal))
                feats.append(mav(ch_signal))
                feats.append(np.std(ch_signal))
                feats.append(np.var(ch_signal))
                feats.append(iav(ch_signal))
                feats.append(mean_frequency(ch_signal))
                # feats.extend(wavelet_features(ch_signal, level=3))

            features.append(feats)
            targets.append(parse_pressure_label(label))

    cols = [f"{ch}_{feat}" for ch in channels for feat in feature_names]
    X = pd.DataFrame(features, columns=cols)
    y = pd.DataFrame(targets, columns=["left", "front", "right"])

    return X, y



# Extract features for train and test
X, y = extract_features(filtered)

print("--- Train:")
print(X.head())
print(y)


--- Train:
   ch_1_RMS  ch_1_RMS_SD  ch_1_ZC     ch_1_WL  ch_1_MAV  ch_1_STD  ch_1_VAR  \
0  5.122753     2.485712        0  819.992087  4.479267  2.485712  6.178763   
1  4.998778     2.470103        0  822.096215  4.345846  2.470103  6.101409   
2  5.597282     2.905844        0  883.433145  4.783894  2.905844  8.443929   
3  5.235488     2.615958        0  897.583967  4.535096  2.615958  6.843234   
4  4.916792     2.291235        0  793.165084  4.350297  2.291235  5.249759   

      ch_1_IAV   ch_1_MF  ch_2_RMS  ...   ch_2_MF  ch_3_RMS  ch_3_RMS_SD  \
0  1679.725132  5.718110  4.404497  ...  5.455315  3.791556     1.817438   
1  1629.692073  5.508347  4.840424  ...  4.778621  4.454717     2.232283   
2  1793.960159  5.539247  5.063685  ...  5.423731  4.198375     1.992627   
3  1700.661047  6.335879  4.751806  ...  6.028689  4.417330     2.057056   
4  1631.361334  5.312857  4.278426  ...  5.112776  4.079766     1.899633   

   ch_3_ZC     ch_3_WL  ch_3_MAV  ch_3_STD  ch_3_VAR     

# Classification

## Random Forest Tree

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Train-test -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline -------------------------------------------
steps = []

if USE_SCALING:
    print("Scaler created")
    steps.append(('scaler', StandardScaler()))

steps.append(('regressor', MultiOutputRegressor(RandomForestRegressor(random_state=42))))

pipeline = Pipeline(steps)

# Grid search -----------------------------------------
param_grid = {}

param_grid.update({
    'regressor__estimator__n_estimators': [50, 100],
    'regressor__estimator__max_depth': [None, 10, 20],
    'regressor__estimator__min_samples_split': [2, 5],
})

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Predict --------------------------------------------
grid_search.fit(X_train, y_train)
y_pred = grid_search.predict(X_test)

# Evaluation -----------------------------------------
print("\n--- Best Parameters ---")
print(grid_search.best_params_)

print("\n--- R² Scores (per output) ---")
print(r2_score(y_test, y_pred, multioutput='raw_values'))

print("\n--- RMSE (per output) ---")
print(np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values')))

# Export --------------------------------------------
best_pipeline = grid_search.best_estimator_
joblib.dump(best_pipeline, f'models/Frog_pressure_regressor_1.joblib')


Scaler created
Fitting 3 folds for each of 12 candidates, totalling 36 fits

--- Best Parameters ---
{'regressor__estimator__max_depth': 20, 'regressor__estimator__min_samples_split': 2, 'regressor__estimator__n_estimators': 100}

--- R² Scores (per output) ---
[0.46138409 0.3276643  0.47532326]

--- RMSE (per output) ---
[21.20029905 25.06982196 21.31202817]


['models/Frog_pressure_regressor_1.joblib']

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# Create binary labels
th = 55
y_test_binary = np.where(np.any(y_test.values > th, axis=1), "shoot", "rest")
y_pred_binary = np.where(np.any(y_pred > th, axis=1), "shoot", "rest")

y_pred_display = np.rint(y_pred).astype(int)

# Create DataFrame for checking
df_check = pd.DataFrame({
    "y_test": list(y_test.values),
    "y_test_binary": y_test_binary,
    "y_pred": list(y_pred_display),  
    "y_pred_binary": y_pred_binary
})


# Print first 50 rows
print(df_check.head(50))

print("Binary Classification Report:")
print(classification_report(y_test_binary, y_pred_binary))

         y_test y_test_binary        y_pred y_pred_binary
0     [0, 0, 0]          rest   [10, 11, 1]          rest
1    [0, 50, 0]          rest    [6, 33, 8]          rest
2    [50, 0, 0]          rest  [19, 10, 14]          rest
3   [0, 100, 0]         shoot  [45, 28, 14]          rest
4    [0, 0, 25]          rest     [0, 9, 4]          rest
5    [0, 50, 0]          rest  [20, 11, 11]          rest
6     [0, 0, 0]          rest     [0, 8, 0]          rest
7    [0, 0, 50]          rest   [20, 10, 8]          rest
8    [50, 0, 0]          rest    [55, 7, 0]          rest
9     [0, 0, 0]          rest    [0, 14, 0]          rest
10    [0, 0, 0]          rest   [40, 13, 2]          rest
11   [25, 0, 0]          rest   [14, 15, 6]          rest
12   [0, 25, 0]          rest    [2, 14, 0]          rest
13   [50, 0, 0]          rest   [54, 11, 2]          rest
14   [0, 50, 0]          rest    [4, 31, 1]          rest
15    [0, 0, 0]          rest    [2, 12, 2]          rest
16  [100, 0, 0